<h1>Ideal gas calculator</h1> 

<h3>Python code 2-01 accompanying <i>Molecular Machines</i></h3> 

<p> G. Woehlke, Mar. 2026  (converted from Wolfram Mathematica, aided by A.I.)</p>

<p>Together with the Boltzmann velocity distribution, the ideal gas model allows calculation of a number of parameters, such as molar volume, kinetic energy of each particle, density (or distance) of particles, and others:

In [2]:
import numpy as np
import pandas as pd
from pint import UnitRegistry
import ipywidgets as widgets
from IPython.display import display

ureg = UnitRegistry()
Q_ = ureg.Quantity

# Constants
nA = Q_(6.02214199e23, 1 / ureg.mole)
kB = Q_(1.380649e-23, ureg.joule / ureg.kelvin)
Rgas = nA * kB  # ideal gas constant

# Molecular masses (kg per molecule)
masses = {
    "H2": Q_(2.01588e-3, ureg.kg / ureg.mole) / nA,
    "He": Q_(4.002602e-3, ureg.kg / ureg.mole) / nA,
    "N2": Q_(28.0134e-3, ureg.kg / ureg.mole) / nA,
    "O2": Q_(31.998e-3, ureg.kg / ureg.mole) / nA,
    "Ar": Q_(39.948e-3, ureg.kg / ureg.mole) / nA,
}

def fmt(q, unit=None, digits=2):
    if unit:
        q = q.to(unit)
    return f"{q.magnitude:.{digits}f} {q.units}"
    
def compute(m, amt, T, p):
    Tq = Q_(T, ureg.kelvin)
    pq = Q_(p, ureg.millibar).to(ureg.pascal)

    # Temperature in Celsius
    T_C = Tq.to(ureg.degC)

    # Volume
    V = (Q_(amt, ureg.mole) * Rgas * Tq / pq).to(ureg.liter)

    # Particle distance
    delta_s = ((kB * Tq / pq) ** (1/3)).to(ureg.meter)

    # Internal energy
    U = ((3/2) * kB * Tq).to(ureg.joule)
    
    # Mean velocity
    v_mean = np.sqrt((8 * kB * Tq) / (np.pi * m)).to(ureg.meter / ureg.second)

    # Most likely velocity
    v_lky = np.sqrt((2 * kB * Tq) / (m)).to(ureg.meter / ureg.second)

    # Collision frequency
    collision_freq = (v_mean / delta_s).to(1 / ureg.second)

    df = pd.DataFrame([
        ["Temperature °C", "T - 273.15", fmt(T_C, ureg.degC)],
        ["Volume", "nRT/p", fmt(V, ureg.liter)],
        ["Particle distance", "(n/V)^(-1/3)", fmt(delta_s, ureg.nanometer)],
        ["Kinetic energy", "3/2 kB T", U],
        ["Mean velocity", "sqrt(8kBT/πm)", fmt(v_mean, ureg.meter/ureg.second)],
        ["Most likely velocity", "sqrt(2kBT/m)", fmt(v_lky, ureg.meter/ureg.second)],
        ["Collision frequency", "⟨v⟩/Δs", fmt(collision_freq, ureg.gigahertz)],
    ], columns=["Parameter", "Calculation", "Value"])

    return df

# Interactive widgets
element_widget = widgets.Dropdown(
    options=list(masses.keys()),
    value="He",
    description="Molecule "
)

amt_widget = widgets.FloatSlider(
    value=1.0, min=0.1, max=10, step=0.1,
    description="Moles"
)

T_widget = widgets.FloatSlider(
    value=273.1, min=100, max=800,
    description="Kelvins"
)

p_widget = widgets.FloatSlider(
    value=1013, min=100, max=10000,
    description="Millibars"
)

ui = widgets.VBox([element_widget, amt_widget, T_widget, p_widget])

out = widgets.interactive_output(
    lambda Gas, Moles, Temp, Pressure: display(
        compute(masses[Gas], Moles, Temp, Pressure)
    ),
    {
        "Gas": element_widget,
        "Moles": amt_widget,
        "Temp": T_widget,
        "Pressure": p_widget,
    }
)

display(ui, out)

Output()

<h3>Legend </h3>
<p>Calculations for ideal gases. The calculator uses the velocity distribution presented in <i>2-01_Boltzmann.ipynb</i> to calculate molecule distances, kinetic energy and more.</p>
<p><span style="color:blue">Compare</span> the average free travel distances with molecular distances in liquids such as water! </p>